<a href="https://colab.research.google.com/github/BozNatanium/AI_NLP_labs/blob/main/rnn_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: генерация текста с помощью RNN (собственный корпус)

## Задача

1. Собрать свой небольшой текстовый корпус (не менее 1000 предложений) с помощью библиотеки `requests` (парсинг новостного сайта, блога, форума и т.д.).
2. Обучить рекуррентную нейросеть (RNN/LSTM) на собранных данных для генерации текста (по образцу из приложенного ноутбука `Copy_of_rnn.ipynb`).
3. После обучения вывести на экран 2–3 сгенерированных предложения.
4. *Дополнительно (на 5 баллов, но не обязательно):* посчитать метрику перплексии (perplexity) на валидационной выборке.
5. *Для себя (не оценивается):* обучить модель с использованием GPU.

## Критерии оценки

- **3 балла** — корпус собран (≥1000 предложений), модель обучена, сгенерировано хотя бы 1 предложение.
- **4 балла** — всё из п.3 + код с комментариями, объясняющими ключевые этапы.
- **5 баллов** — всё из п.4 + дополнительно посчитана метрика перплексии.

## Важно

- Качество сгенерированного текста не оценивается.
- Выберите **свой уникальный сайт** для парсинга и укажите его в отчёте.
- Не используйте готовые датасеты из интернета.


In [22]:
!pip install beautifulsoup4 requests lxml -q

import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow версия:", tf.__version__)
print("GPU доступна:", tf.config.list_physical_devices('GPU'))

def scrape_corpus(base_url, num_pages=5):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    texts = []

    for page in range(1, num_pages + 1):
        url = f"{base_url}?page={page}"
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # Пример для заголовков новостей:
            for tag in soup.find_all(['h2', 'h3', 'p']):
                text = tag.get_text(strip=True)
                if text and len(text) > 10:
                    texts.append(text)

            print(f"Страница {page}: собрано {len(texts)} текстов")
            time.sleep(1)  # вежливый парсинг

        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")

    return texts


# взял сайт википедии
base_url = "https://ru.wikipedia.org/wiki/%D0%92%D0%B8%D0%BA%D0%B8%D0%BF%D0%B5%D0%B4%D0%B8%D1%8F"
corpus = scrape_corpus(base_url, num_pages=5)

print(f"\nВсего собрано текстов: {len(corpus)}")
print("Примеры:", corpus[:5])

# токенятник
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

# Преобразуем тексты в последовательности чисел
sequences = tokenizer.texts_to_sequences(corpus)

# Создаём входные и выходные данные для causal language modeling
X, y = [], []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

# Паддинг
X = pad_sequences(X)

# One-hot encoding для y
vocab_size = len(tokenizer.word_index) + 1
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

print(f"Размер входных данных X: {X.shape}")
print(f"Размер выходных данных y: {y.shape}")
print(f"Размер словаря: {vocab_size}")

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=100, input_length=X.shape[1]))
model.add(LSTM(150, return_sequences=False))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Обучение (эпохи оставлял как есть, не увеличивал)
history = model.fit(X, y, epochs=10, batch_size=32, validation_split=0.2)

def generate_text(seed_text, next_words, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Пример генерации
seed = "Википедия"
generated = generate_text(seed, next_words=5, max_sequence_len=X.shape[1])
print("Сгенерированный текст:", generated)

# еще 2 примера
seed2 = "Цель"
generated2 = generate_text(seed2, next_words=5, max_sequence_len=X.shape[1])
print("Сгенерированный текст 2:", generated2)

seed3 = "Энциклопедия"
generated3 = generate_text(seed3, next_words=5, max_sequence_len=X.shape[1])
print("Сгенерированный текст 3:", generated3)

# Оцениваем модель на валидационных данных
loss, accuracy = model.evaluate(X, y, verbose=0)
perplexity = np.exp(loss)

print(f"Потери (loss): {loss:.4f}")
print(f"Перплексия: {perplexity:.4f}")

TensorFlow версия: 2.20.0
GPU доступна: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Страница 1: собрано 103 текстов
Страница 2: собрано 206 текстов
Страница 3: собрано 309 текстов
Страница 4: собрано 412 текстов
Страница 5: собрано 515 текстов

Всего собрано текстов: 515
Примеры: ['Википе́дия(англ.Wikipedia, произноситсяо\xa0файле/ˌwɪkɪˈpiːdiə/,разг.«Ви́ки»[6]по названию технологиивеб-сайта)\xa0—многоязычнаяобщедоступнаяинтернет-энциклопедиясосвободным контентоми элементамисоциальной сети, поддержку и написание которой осуществляютдобровольцы— «википедисты», посредствомоткрытого сотрудничестваи с использованием программного обеспечения (сайта)MediaWiki, а также системы редактирования на основевики-принципов. Википедия является самым крупным и, в то же время, наиболее читаемымсправочником[7], а также самой полной энциклопедией из когда-либо создававшихся за всюисторию человечества[8][9]. По состоянию на май 2025\xa0года сайт находился на 8 месте по посещаемости в 

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.0385 - loss: 7.2426 - val_accuracy: 0.0442 - val_loss: 6.4407
Epoch 2/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.0520 - loss: 5.9646 - val_accuracy: 0.1092 - val_loss: 4.9699
Epoch 3/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.1543 - loss: 4.5936 - val_accuracy: 0.3794 - val_loss: 3.6604
Epoch 4/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.3932 - loss: 3.3459 - val_accuracy: 0.6093 - val_loss: 2.5337
Epoch 5/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.6189 - loss: 2.2920 - val_accuracy: 0.7615 - val_loss: 1.6772
Epoch 6/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.7775 - loss: 1.5027 - val_accuracy: 0.8711 - val_loss: 1.0623
Epoch 7/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.8833 - loss: 0.9559 - val_accuracy: 0.9385 - val_loss: 0.6637
Epoch 8/10
668/668 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9431 - loss: 0.5932 - 